# Broomstick Model - ENDF/B-VIII.1

This notebook compares YAMC and OpenMC simulation results for a broomstick (cylinder) geometry using the ENDF/B-VIII.1 nuclear data library for photon transport.

## Model Description

- **Geometry**: Cylinder with radius 1 cm, height 100 cm (z = -50 to +50), vacuum boundaries
- **Source**: Point source at origin, 1 MeV monoenergetic photons, isotropic
- **Material**: Single element at 1.0 g/cm³, temperature 294 K
- **Statistics**: 5,000,000 particles × 10 batches = 50,000,000 total histories
- **Tallies**: Photon flux (energy-binned, CCFE-24-PHOTON), coherent-scatter, incoherent-scatter, photoelectric, pair-production

In [1]:
# Setup and imports
import numpy as np
import pandas as pd
import glob
import os
import yamc
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

# Configuration
NUCLEAR_DATA_DIR = {
    "endf-b8.1": "/home/jon/nuclear_data/endf-b8.1-hdf5/endfb-viii.1-hdf5/",
}

NUCLEAR_DATA = "endf-b8.1"
PARTICLES = 5000000
BATCHES = 10
SOURCE_ENERGY = 1e6  # 1 MeV photons
MAX_PLOT_ENERGY = SOURCE_ENERGY * 1.1
FLUX_DECADES = 6

# Optional smoke-test cap on the iteration list (env var; unset/0 = full).
NUMBER_OF_SIMULATIONS = int(os.environ.get("YAMC_VERIFY_NUMBER_OF_SIMULATIONS", "0")) or None

# Cylinder dimensions
CYLINDER_RADIUS = 1.0  # cm
CYLINDER_HALF_HEIGHT = 50.0  # cm (total height = 100 cm)

# CCFE-24-PHOTON energy bins (eV)
PHOTON_BINS = [
    1.0e3, 5.0e3, 1.0e4, 2.0e4, 5.0e4, 1.0e5, 2.0e5, 3.0e5, 4.0e5,
    5.0e5, 5.11e5, 6.0e5, 8.0e5, 1.0e6, 1.5e6, 2.0e6, 3.0e6, 4.0e6,
    5.0e6, 6.0e6, 8.0e6, 1.0e7, 1.2e7, 1.5e7, 2.0e7,
]

In [2]:
def run_openmc_simulation(element, nuclear_data):
    """Run OpenMC photon simulation for a single element."""
    import openmc as mc

    mc.reset_auto_ids()
    mc.config['cross_sections'] = f"{NUCLEAR_DATA_DIR[nuclear_data]}cross_sections.xml"

    # Geometry: broomstick cylinder
    cylinder = mc.ZCylinder(r=CYLINDER_RADIUS, boundary_type='vacuum')
    z_bot = mc.ZPlane(z0=-CYLINDER_HALF_HEIGHT, boundary_type='vacuum')
    z_top = mc.ZPlane(z0=CYLINDER_HALF_HEIGHT, boundary_type='vacuum')
    region = -cylinder & +z_bot & -z_top

    # Material
    material = mc.Material()
    material.add_element(element, 1.0)
    material.set_density("g/cm3", 1.0)
    material.temperature = 294

    cell = mc.Cell(name="broomstick", region=region, fill=material)
    geometry = mc.Geometry([cell])

    # Source: 1 MeV isotropic photons at origin
    source = mc.IndependentSource(
        space=mc.stats.Point([0, 0, 0]),
        energy=mc.stats.Discrete([SOURCE_ENERGY], [1.0]),
        angle=mc.stats.Isotropic(),
    )
    source.particle = "photon"

    settings = mc.Settings(
        particles=PARTICLES,
        batches=BATCHES,
        source=source,
        seed=1,
        run_mode='fixed source',
        output={'tallies': False, 'summary': False},
    )

    # Filters and tallies
    cell_filter = mc.CellFilter(cell)
    photon_filter = mc.ParticleFilter("photon")
    energy_filter = mc.EnergyFilter(PHOTON_BINS)
    energy_bins_2d = energy_filter.bins
    energy_bins = np.concatenate([[energy_bins_2d[0, 0]], energy_bins_2d[:, 1]])

    tallies = []

    # Photon interaction score tallies
    photon_scores = ['coherent-scatter', 'incoherent-scatter', 'photoelectric', 'pair-production']
    for score in ['flux'] + photon_scores:
        tally = mc.Tally()
        tally.filters = [cell_filter, photon_filter]
        tally.scores = [score]
        tally.name = score
        tallies.append(tally)

    # Energy-binned photon flux tally
    tally_spectrum = mc.Tally()
    tally_spectrum.filters = [cell_filter, photon_filter, energy_filter]
    tally_spectrum.scores = ['flux']
    tally_spectrum.name = "flux_energy_binned"
    tallies.append(tally_spectrum)

    # Run
    model = mc.Model(geometry=geometry, settings=settings, tallies=tallies)
    sp_filename = model.run(apply_tally_results=True)
    sp = mc.StatePoint(sp_filename)

    transport_time = sp.runtime['transport']
    total_particles = PARTICLES * BATCHES
    particles_per_second = total_particles / transport_time

    results = {
        'element': element,
        'code': 'openmc',
        'particles/s': particles_per_second,
        'tallies': {},
        'tally_std': {},
    }

    for tally in tallies:
        if tally.name != "flux_energy_binned":
            results['tallies'][tally.name] = tally.mean.flatten()[0]
            results['tally_std'][tally.name] = tally.std_dev.flatten()[0]

    results['flux_spectra_mean'] = np.array(tally_spectrum.mean).squeeze()
    results['flux_spectra_std_dev'] = np.array(tally_spectrum.std_dev).squeeze()
    results['energy_bins'] = energy_bins

    return results

In [3]:
def run_yamc_simulation(element, nuclear_data):
    """Run YAMC photon simulation for a single element. Returns (results_dict, model)."""
    import yamc as mc

    photon_path = f"{NUCLEAR_DATA_DIR[nuclear_data]}photon/{element}.h5"

    # Geometry: broomstick cylinder
    cylinder = mc.ZCylinder(radius=CYLINDER_RADIUS, boundary='vacuum')
    z_bot = mc.ZPlane(z0=-CYLINDER_HALF_HEIGHT, boundary='vacuum')
    z_top = mc.ZPlane(z0=CYLINDER_HALF_HEIGHT, boundary='vacuum')
    region = -cylinder & +z_bot & -z_top

    # Material - photon-only, no neutron data needed
    material = mc.Material(
        composition={element: 1.0},
        density=1.0,
        temperature=294,
        id=1,
    )
    material.read_nuclear_data('endf-b8.1')

    cell = mc.Cell(id=1, name="broomstick", region=region, material=material)
    geometry = mc.Geometry([cell])

    # Source: 1 MeV isotropic photons at origin
    source = mc.PhotonSource(
        position=mc.PointSource(0, 0, 0),
        direction=mc.Isotropic(),
        energy=mc.Discrete([SOURCE_ENERGY], [1.0]),
    )

    tallies = []

    # Photon interaction score tallies
    photon_scores = ['coherent-scatter', 'incoherent-scatter', 'photoelectric', 'pair-production']
    for score in ['flux'] + photon_scores:
        t = mc.Tally(
            scores=[str(score)],
            name=str(score),
            cell=cell,
            particle="photon",
        )
        tallies.append(t)

    # Energy-binned photon flux tally
    tally_spectrum = mc.Tally(
        scores=['flux'],
        name="flux_energy_binned",
        cell=cell,
        particle="photon",
        energy_group_structure='CCFE-24-PHOTON',
    )
    tallies.append(tally_spectrum)

    # Energy group bins (for plotting)
    energy_bins = np.array(tally_spectrum.energy_bins)

    # Run
    model = mc.Model(
        geometry=geometry,
        tallies=tallies,
        source=source,
        particles_per_batch=PARTICLES,
        batches=BATCHES,
        seed=1,
        photon_transport=True,
    )
    sim_results = model.simulate()

    results = {
        'element': element,
        'code': 'yamc',
        'particles/s': model.particles_per_second,
        'tallies': {},
        'tally_std': {},
    }

    for t in tallies:
        if t.name != "flux_energy_binned":
            r = sim_results[t]
            results['tallies'][t.name] = r.mean[0]
            results['tally_std'][t.name] = r.standard_deviation[0]

    spec = sim_results[tally_spectrum]
    results['flux_spectra_mean'] = np.array(spec.mean).squeeze()
    results['flux_spectra_std_dev'] = np.array(spec.standard_deviation).squeeze()
    results['energy_bins'] = energy_bins

    return results, model

In [4]:
def create_step_trace(energy_bins, values, name, color, dash=None):
    """Create a step plot trace for Plotly (similar to matplotlib step with where='post')."""
    x_step = []
    y_step = []
    for i in range(len(values)):
        x_step.extend([energy_bins[i], energy_bins[i+1]])
        y_step.extend([values[i], values[i]])

    return go.Scatter(
        x=x_step,
        y=y_step,
        mode='lines',
        name=name,
        line=dict(color=color, width=2, dash=dash),
    )


def create_comparison_plot(openmc_results, yamc_results):
    """Create a Plotly subplot with photon flux spectra (top) and ratio (bottom)."""

    energy_bins = openmc_results['energy_bins']
    bin_centers = np.sqrt(energy_bins[:-1] * energy_bins[1:])

    openmc_flux = openmc_results['flux_spectra_mean'].copy()
    openmc_std = openmc_results['flux_spectra_std_dev'].copy()
    yamc_flux = yamc_results['flux_spectra_mean'].copy()
    yamc_std = yamc_results['flux_spectra_std_dev'].copy()

    # Normalize by lethargy bin width for per-unit-lethargy spectrum
    # Equivalent to openmc.EnergyFilter.lethargy_bin_width / yamc.EnergyFilter.lethargy_bin_width
    lethargy_bin_width = np.log10(energy_bins[1:] / energy_bins[:-1])
    openmc_flux /= lethargy_bin_width
    openmc_std /= lethargy_bin_width
    yamc_flux /= lethargy_bin_width
    yamc_std /= lethargy_bin_width

    # Find the lowest energy bin with non-zero flux in either code
    has_flux = (openmc_flux > 0) | (yamc_flux > 0)
    if np.any(has_flux):
        first_nonzero_idx = np.argmax(has_flux)
        min_plot_energy = energy_bins[first_nonzero_idx]
    else:
        min_plot_energy = energy_bins[0]

    max_flux = max(
        np.max(openmc_flux[openmc_flux > 0]) if np.any(openmc_flux > 0) else 1e-10,
        np.max(yamc_flux[yamc_flux > 0]) if np.any(yamc_flux > 0) else 1e-10
    )
    flux_threshold = max_flux / (10 ** FLUX_DECADES)

    openmc_flux_plot = np.where(openmc_flux >= flux_threshold, openmc_flux, np.nan)
    yamc_flux_plot = np.where(yamc_flux >= flux_threshold, yamc_flux, np.nan)
    openmc_std_plot = np.where(openmc_flux >= flux_threshold, openmc_std, np.nan)
    yamc_std_plot = np.where(yamc_flux >= flux_threshold, yamc_std, np.nan)

    # Calculate ratio with error propagation
    valid = (openmc_flux > 0) & (yamc_flux > 0)
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio = np.where(valid, yamc_flux / openmc_flux, np.nan)
        rel_err_yamc = np.where(yamc_flux > 0, yamc_std / yamc_flux, 0)
        rel_err_openmc = np.where(openmc_flux > 0, openmc_std / openmc_flux, 0)
    ratio_rel_err = np.sqrt(rel_err_yamc**2 + rel_err_openmc**2)
    ratio_std = np.where(valid, ratio * ratio_rel_err, np.nan)

    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.08,
        row_heights=[0.7, 0.3],
        subplot_titles=('Photon Flux per Unit Lethargy', 'YAMC / OpenMC Ratio')
    )

    fig.add_trace(
        create_step_trace(energy_bins, openmc_flux_plot, 'OpenMC', 'blue'),
        row=1, col=1
    )
    fig.add_trace(
        create_step_trace(energy_bins, yamc_flux_plot, 'YAMC', 'red', dash='dash'),
        row=1, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=bin_centers, y=openmc_flux_plot, mode='markers',
            name='OpenMC error',
            marker=dict(color='blue', size=4, symbol='circle'),
            error_y=dict(type='data', array=openmc_std_plot, visible=True, color='blue', thickness=1),
            showlegend=False,
        ),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(
            x=bin_centers, y=yamc_flux_plot, mode='markers',
            name='YAMC error',
            marker=dict(color='red', size=4, symbol='square'),
            error_y=dict(type='data', array=yamc_std_plot, visible=True, color='red', thickness=1),
            showlegend=False,
        ),
        row=1, col=1
    )

    fig.add_trace(
        create_step_trace(energy_bins, ratio, 'YAMC/OpenMC', 'green'),
        row=2, col=1
    )
    fig.add_trace(
        go.Scatter(
            x=np.concatenate([bin_centers, bin_centers[::-1]]),
            y=np.concatenate([ratio + ratio_std, (ratio - ratio_std)[::-1]]),
            fill='toself',
            fillcolor='rgba(0, 128, 0, 0.2)',
            line=dict(color='rgba(255,255,255,0)'),
            name='Uncertainty',
            showlegend=False,
        ),
        row=2, col=1
    )
    fig.add_hline(y=1.0, line_dash="dash", line_color="gray", row=2, col=1)

    fig.update_xaxes(type="log", title_text="Energy (eV)",
                     range=[np.log10(min_plot_energy), np.log10(MAX_PLOT_ENERGY)],
                     exponentformat='e', row=2, col=1)
    fig.update_xaxes(type="log",
                     range=[np.log10(min_plot_energy), np.log10(MAX_PLOT_ENERGY)],
                     exponentformat='e', row=1, col=1)
    fig.update_yaxes(type="log", title_text="Photon Flux / Lethargy", exponentformat='e', row=1, col=1)
    fig.update_yaxes(title_text="Ratio", range=[0.5, 1.5], row=2, col=1)

    fig.update_layout(
        height=700,
        legend=dict(x=0.02, y=0.98),
        margin=dict(l=60, r=20, t=40, b=40)
    )

    return fig


def calculate_spectral_agreement(openmc_results, yamc_results, flux_threshold=1e-6):
    """Calculate statistical agreement metrics for photon flux spectra comparison."""

    openmc_flux = openmc_results['flux_spectra_mean']
    openmc_std = openmc_results['flux_spectra_std_dev']
    yamc_flux = yamc_results['flux_spectra_mean']
    yamc_std = yamc_results['flux_spectra_std_dev']

    chi_squared_terms = []
    rel_diffs = []
    weighted_rel_diffs = []
    flux_weights = []

    for i in range(len(openmc_flux)):
        omc = openmc_flux[i]
        ymc = yamc_flux[i]
        omc_std = openmc_std[i]
        ymc_std = yamc_std[i]

        combined_std = np.sqrt(omc_std**2 + ymc_std**2)
        if combined_std > 0:
            chi_squared_terms.append(((ymc - omc) / combined_std)**2)

        if omc > flux_threshold or ymc > flux_threshold:
            if omc > 0:
                rel_diff = abs((ymc - omc) / omc) * 100
                if np.isfinite(rel_diff):
                    rel_diffs.append(rel_diff)
                    weight = max(omc, ymc)
                    weighted_rel_diffs.append(rel_diff * weight)
                    flux_weights.append(weight)

    chi_squared = np.sum(chi_squared_terms)
    dof = len(chi_squared_terms)
    reduced_chi_squared = chi_squared / dof if dof > 0 else 0.0

    median_rel_diff = np.median(rel_diffs) if rel_diffs else 0.0
    weighted_avg_rel_diff = np.sum(weighted_rel_diffs) / np.sum(flux_weights) if flux_weights else 0.0

    if reduced_chi_squared < 1.5:
        quality = "Excellent"
    elif reduced_chi_squared < 3.0:
        quality = "Good"
    elif reduced_chi_squared < 5.0:
        quality = "Fair"
    else:
        quality = "Poor"

    return {
        'reduced_chi_squared': reduced_chi_squared,
        'degrees_of_freedom': dof,
        'median_rel_diff': median_rel_diff,
        'weighted_avg_rel_diff': weighted_avg_rel_diff,
        'quality': quality,
        'bins_analyzed': len(rel_diffs),
        'total_bins': len(openmc_flux)
    }


def create_agreement_table(stats):
    """Create a summary table of spectral agreement statistics."""
    rows = [
        {'Metric': 'Reduced Chi-Squared', 'Value': f"{stats['reduced_chi_squared']:.2f}",
         'Note': f"(ideal ~ 1.0, dof={stats['degrees_of_freedom']})"},
        {'Metric': 'Weighted Avg Rel. Diff', 'Value': f"{stats['weighted_avg_rel_diff']:.2f}%",
         'Note': '(flux-weighted)'},
        {'Metric': 'Median Rel. Diff', 'Value': f"{stats['median_rel_diff']:.2f}%",
         'Note': '(robust to outliers)'},
        {'Metric': 'Agreement Quality', 'Value': stats['quality'],
         'Note': f"({stats['bins_analyzed']}/{stats['total_bins']} bins analyzed)"},
    ]
    return pd.DataFrame(rows)


# Photon interaction MT descriptions
PHOTON_MT_DESCRIPTIONS = {
    'flux': 'Total photon flux',
    'coherent-scatter': 'Coherent (Rayleigh) scatter (MT 502)',
    'incoherent-scatter': 'Incoherent (Compton) scatter (MT 504)',
    'photoelectric': 'Photoelectric absorption (MT 522)',
    'pair-production': 'Pair production (MT 516)',
}


def create_tally_table(openmc_results, yamc_results):
    """Create a comparison table of photon tallies."""

    all_tallies = set(openmc_results['tallies'].keys()) | set(yamc_results['tallies'].keys())
    rows = []

    for name in sorted(all_tallies):
        openmc_val = openmc_results['tallies'].get(name, np.nan)
        openmc_std = openmc_results['tally_std'].get(name, np.nan)
        yamc_val = yamc_results['tallies'].get(name, np.nan)
        yamc_std = yamc_results['tally_std'].get(name, np.nan)

        if (openmc_val == 0 or np.isnan(openmc_val)) and (yamc_val == 0 or np.isnan(yamc_val)):
            continue

        if openmc_val != 0 and not np.isnan(openmc_val):
            rel_diff = (yamc_val - openmc_val) / openmc_val * 100
        else:
            rel_diff = np.nan

        reaction = PHOTON_MT_DESCRIPTIONS.get(name, name)

        rows.append({
            'Score': name,
            'Reaction': reaction,
            'OpenMC': f"{openmc_val:.4e} +/- {openmc_std:.2e}",
            'YAMC': f"{yamc_val:.4e} +/- {yamc_std:.2e}",
            'Rel. Diff (%)': f"{rel_diff:.2f}%" if not np.isnan(rel_diff) else 'N/A',
        })

    return pd.DataFrame(rows)

In [5]:
# Discover elements with photon data available
photon_dir = os.path.join(NUCLEAR_DATA_DIR[NUCLEAR_DATA], 'photon')

photon_files = sorted(glob.glob(os.path.join(photon_dir, '*.h5')))
elements = [os.path.splitext(os.path.basename(f))[0] for f in photon_files]

if not elements:
    raise FileNotFoundError(
        f"No photon data files found in {photon_dir}\n"
        f"Please check that the photon/ subdirectory exists."
    )

# Filter to elements that YAMC can construct from natural abundance.
# Synthetic/short-lived elements (Tc, Pm, Po, At, Rn, Fr, Ra, Ac, Np, Pu, Am, Cm, Bk, Cf, Es, Fm)
# have no natural abundance and cannot be expanded by composition={element: 1.0}.
supported = []
unsupported = []
for e in elements:
    try:
        yamc.Material(composition={e: 1.0}, density=1.0)
        supported.append(e)
    except Exception:
        unsupported.append(e)
elements = supported

if NUMBER_OF_SIMULATIONS:
    elements = elements[:NUMBER_OF_SIMULATIONS]

if unsupported:
    print(f"Skipping {len(unsupported)} elements without natural abundance: {unsupported}")
print(f"Running photon simulations for {len(elements)} elements: {elements}")

Skipping 16 elements without natural abundance: ['Ac', 'Am', 'At', 'Bk', 'Cf', 'Cm', 'Es', 'Fm', 'Fr', 'Np', 'Pm', 'Po', 'Pu', 'Ra', 'Rn', 'Tc']
Running photon simulations for 3 elements: ['Ag', 'Al', 'Ar']


In [6]:
# Run all simulations and collect results
all_results = {}
first_yamc_model = None

for element in elements:
    print(f"\nProcessing {element}...")
    try:
        yamc_results, yamc_model = run_yamc_simulation(element, NUCLEAR_DATA)
        if first_yamc_model is None:
            first_yamc_model = yamc_model
    except Exception as e:
        print(f"  YAMC Error: {e}")
        continue

    try:
        openmc_results = run_openmc_simulation(element, NUCLEAR_DATA)
        all_results[element] = {
            'openmc': openmc_results,
            'yamc': yamc_results
        }
        print(f"  OpenMC: {openmc_results['particles/s']:.0f} particles/s")
        print(f"  YAMC:   {yamc_results['particles/s']:.0f} particles/s")
    except Exception as e:
        print(f"  OpenMC Error: {e}")
        continue

print(f"\nCompleted {len(all_results)} / {len(elements)} elements successfully.")


Processing Ag...


  Loaded photon data (Arrow): Z=47, 9273 energy points, 15 subshells, 15 Compton shells
Starting batch 1/10


Starting batch 2/10


Starting batch 3/10


Starting batch 4/10


Starting batch 5/10


... (53 outputs left out)

 Simulating batch 8


 Simulating batch 9


 Simulating batch 10


 Creating state point statepoint.10.h5...

 =======================>     TIMING STATISTICS     <=======================

 Total time for initialization     = 8.5063e-02 seconds
   Reading cross sections          = 8.2504e-02 seconds
 Total time in simulation          = 5.1810e+00 seconds
   Time in transport only          = 4.8906e+00 seconds
   Time in active batches          = 5.1810e+00 seconds
   Time accumulating tallies       = 2.8535e-01 seconds
   Time writing statepoints        = 4.2036e-03 seconds
 Total time for finalization       = 2.2100e-07 seconds
 Total time elapsed                = 5.2664e+00 seconds
 Calculation Rate (active)         = 9.65056e+06 particles/second

 ============================>     RESULTS     <============================

 Leakage Fraction            = 1.00132 +/- 0.00000

  OpenMC: 10223744 particles/s
  YAMC:   7735199 particles/s

Completed 3 / 3 elements successfully.


In [7]:
# Display broomstick model geometry with source from first simulation
import base64
from IPython.display import HTML

if first_yamc_model is not None:
    html_plot = first_yamc_model.plot(basis="xz", n_samples=500, axis_units="cm")
    html_plot.save("broomstick_model_plot.html")
    encoded = base64.b64encode(html_plot.html.encode()).decode()
    display(HTML(f'<iframe src="data:text/html;base64,{encoded}" width="100%" height="600" frameborder="0"></iframe>'))
else:
    print("No successful YAMC simulation to plot.")

/home/jon/yamc-org/yamc-verification/.venv/lib/python3.13/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


## Results by Element

Below are the comparison results for each element. Each section includes:

- Interactive photon flux spectra plot (log-log scale)
- YAMC/OpenMC ratio plot with uncertainty band
- Photon tally comparison table (flux, coherent-scatter, incoherent-scatter, photoelectric, pair-production)
- Performance comparison (particles/second)

In [8]:
# Display results for each element
for element, results in all_results.items():
    openmc_results = results['openmc']
    yamc_results = results['yamc']

    # Header
    display(Markdown(f"## {element}"))

    # Performance comparison
    openmc_pps = openmc_results['particles/s']
    yamc_pps = yamc_results['particles/s']
    speedup = yamc_pps / openmc_pps
    display(Markdown(f"**Performance:** OpenMC: {openmc_pps:,.0f} particles/s | YAMC: {yamc_pps:,.0f} particles/s | Speedup: {speedup:.2f}x"))

    # Flux spectra plot
    fig = create_comparison_plot(openmc_results, yamc_results)
    fig.update_layout(title=f"{element} - Photon Flux Spectra Comparison")
    fig.show()

    # Spectral agreement statistics
    display(Markdown("### Spectral Agreement"))
    stats = calculate_spectral_agreement(openmc_results, yamc_results)
    agreement_table = create_agreement_table(stats)
    display(agreement_table.style.hide(axis='index'))

    # Tally table
    display(Markdown("### Photon Tally Comparison"))
    table = create_tally_table(openmc_results, yamc_results)
    display(table.style.hide(axis='index'))

    print()  # Add spacing between elements

## Ag

**Performance:** OpenMC: 9,838,782 particles/s | YAMC: 8,245,949 particles/s | Speedup: 0.84x

### Spectral Agreement

Metric,Value,Note
Reduced Chi-Squared,1.07,"(ideal ~ 1.0, dof=13)"
Weighted Avg Rel. Diff,0.03%,(flux-weighted)
Median Rel. Diff,0.25%,(robust to outliers)
Agreement Quality,Excellent,(13/24 bins analyzed)


### Photon Tally Comparison

Score,Reaction,OpenMC,YAMC,Rel. Diff (%)
coherent-scatter,Coherent (Rayleigh) scatter (MT 502),3.7245e-03 +/- 3.05e-06,3.7261e-03 +/- 2.64e-06,0.04%
flux,Total photon flux,1.5421e+00 +/- 2.17e-04,1.5420e+00 +/- 1.40e-04,-0.01%
incoherent-scatter,Incoherent (Compton) scatter (MT 504),8.7383e-02 +/- 1.35e-05,8.7379e-02 +/- 8.91e-06,-0.00%
photoelectric,Photoelectric absorption (MT 522),4.5977e-02 +/- 5.95e-05,4.6022e-02 +/- 4.96e-05,0.10%


... (8 outputs left out)

### Spectral Agreement

Metric,Value,Note
Reduced Chi-Squared,2.12,"(ideal ~ 1.0, dof=13)"
Weighted Avg Rel. Diff,0.03%,(flux-weighted)
Median Rel. Diff,0.28%,(robust to outliers)
Agreement Quality,Good,(13/24 bins analyzed)


### Photon Tally Comparison

Score,Reaction,OpenMC,YAMC,Rel. Diff (%)
coherent-scatter,Coherent (Rayleigh) scatter (MT 502),7.0711e-04 +/- 7.19e-07,7.0677e-04 +/- 5.94e-07,-0.05%
flux,Total photon flux,1.5506e+00 +/- 2.23e-04,1.5505e+00 +/- 1.59e-04,-0.01%
incoherent-scatter,Incoherent (Compton) scatter (MT 504),9.1546e-02 +/- 1.52e-05,9.1545e-02 +/- 1.09e-05,-0.00%
photoelectric,Photoelectric absorption (MT 522),8.0334e-03 +/- 1.60e-05,7.9955e-03 +/- 1.22e-05,-0.47%
